## Introduction

Endometrial cancer is the most commonly diagnosed gynaecological cancer in developed countries, and its incidence continues to rise. While genome-wide association studies (GWAS) have identified multiple risk loci, translating these associations into biological insights remains difficult, as the majority of associated variants fall in non-coding regions where their functional effects are unclear.Transcriptome-wide association studies (TWAS) address this limitation by integrating GWAS summary statistics with expression quantitative trait loci (eQTL) data. By using eQTL effect estimates to impute genetically regulated gene expression, TWAS can associate predicted expression levels with disease risk, bridging the gap between non-coding variants and gene function. Multi-tissue extensions such as S-MultiXcan further increase statistical power by leveraging shared genetic regulation across tissues and accounting for cross-tissue correlations, enabling the detection of genes whose eQTL signals are individually modest but consistent across tissues.

This project implements and critically evaluates a variant-to-gene pipeline for endometrial cancer by integrating summary statistics from the endometrial cancer GWAS meta-analysis (GCST006464; O'Mara et al. 2018) with GTEx v8 eQTL-based expression models across six biologically relevant tissues: subcutaneous adipose, visceral omentum adipose, ovary, uterus, vagina, and whole blood. The analysis has three aims: (1) quality control of GWAS summary statistics and estimation of SNP heritability; (2) single-tissue and joint-tissue TWAS using elastic net prediction models, with replication of Kho et al. (2021); and (3) a critical comparison of elastic net and mashr prediction models to evaluate how model choice affects tissue-specific gene prioritization.

## Results

### GWAS Quality Control
Of 9,525,227 harmonized SNPs obtained from the GWAS Catalog, 7,112,203 variants passed sequential QC filters and were retained for downstream analysis. Filters applied included the removal of missing essential fields, restriction to autosomal biallelic SNPs, exclusion of palindromic SNPs, and removal of duplicate positions. The MHC region (chr6: 26–34 Mb) was excluded due to its complex LD structure, consistent with Kho et al. (2021) (Table 1).

### Genomic Inflation and SNP Heritability
Genomic inflation assessed via the λ GC statistic (λ GC = 1.097) was consistent with the LDSC-estimated value of 1.108, and is within the acceptable range for a large-scale GWAS meta-analysis (N = 121,885). The Manhattan plot revealed multiple genome-wide significant loci, and the QQ plot demonstrated close adherence to the null diagonal across the bulk of SNPs with departure at the tail consistent with true polygenic signal (Figure 1). SNP heritability estimated via LD Score Regression using Pan-UKBB European LD scores was h2 SNP = 0.0247 (SE = 0.0046). The LDSC intercept of 1.066 (SE = 0.008, ratio = 0.514) suggests mild residual inflation, likely reflecting genuine polygenicity and residual population stratification across contributing studies.

### Independent Loci and Replication
LD clumping identified 16 independent genome-wide significant loci (P < 5×10-8). To validate the processed summary statistics, we assessed whether genome-wide significant signals were present within the cytogenetic band boundaries corresponding to the seven loci reported by Kho et al. (2021) in their S-MultiXcan analysis. Six of the seven loci replicated at genome-wide significance (P < 5×10-8), and all seven showed at least suggestive evidence of association (P < 1×10-5). The one locus that did not reach genome-wide significance was 3q21.3 (EEFSEC; best SNP P = 9.95×10-7) (Table 2), consistent with EEFSEC having been identified through transcriptome-wide association rather than GWAS. The underlying GWAS signal at this locus is sub-threshold, and it is the aggregation of eQTL evidence across tissues that elevates it to significance in the TWAS framework.

![manhattan_qq_plot](./results/aim1/manhattan_qq.png)

<!-- <img src="results/aim1/manhattan_qq.png" width="700"/> -->



Figure 1. Manhattan and Quantile-Quantile (QQ) plots for the endometrial GWAS after QC. Manhattan plot of 7,112,203 QC-passed variants across chromosomes 1–22. The red dashed line indicates the genome-wide significance threshold (P 5×10-8) and the yellow dotted line indicates the suggestive threshold (P 1×10-5). Lead SNPs at genome-wide significant loci are labelled. QQ plot of observed versus expected −log10(P) values. The genomic inflation factor (λ GC = 1.097) indicates modest inflation consistent with polygenicity in a large-sample GWAS rather than systematic bias.

Table 1. GWAS QC filter summary (GCST006464).

| QC Filter | SNPs Remaining | SNPs Removed |
|---|---:|---:|
| Raw input | 9,525,227 | — |
| After dropping missing fields | 9,476,314 | 48,913 |
| After keeping autosomes (chr 1–22) | 9,476,314 | 0 |
| After keeping biallelic SNPs | 8,485,482 | 990,832 |
| After removing palindromic SNPs | 7,162,395 | 1,323,087 |
| After removing MHC region (chr6: 26–34 Mb) | 7,112,226 | 50,169 |
| After removing duplicate positions | 7,112,203 | 23 |
| After SE/p-value sanity checks | 7,112,203 | 0 |
| **Final QC-passed** | **7,112,203** | **2,413,024** |

Table 2. Replication of Kho et al. (2021) S-MultiXcan loci.
Band boundaries from UCSC Genome Browser (GRCh38). GW significant = P < 5×10-8; Suggestive = P < 1×10-5.

| Locus | Gene | Kho et al. P | Proj Best SNP | Proj P | Replication |
|---|---|---|---|---|---|
| 3q21.3 | EEFSEC | 1.10×10-6 | rs872267 | 9.95×10-7 | Suggestive |
| 6q22.31 | HEY2 | 9.94×10-9 | rs2747716 | 2.91×10-10 | GW significant  |
| 15q21.2 | GLDN | 1.34×10-12 | rs17601876 | 3.27×10-14 | GW significant |
| 15q21.2 | CYP19A1 | 9.52×10-12 | rs17601876 | 3.27×10-14 | GW significant |
| 17q11.2 | EVI2A | 1.50×10-6 | rs1129506 | 4.29×10-8 | GW significant |
| 17q21.32 | SKAP1 | 7.27×10-9 | rs882380 | 4.66×10-9 | GW significant |
| 17q21.32 | SNX11 | 5.41×10-7 | rs882380 | 4.66×10-9 | GW significant |

**Note:**   GLDN and CYP19A1 are co-located within the same 15q21.2 locus and share the same index SNP.  
        SKAP1 and SNX11 are co-located within the same 17q21.32 locus and share the same index SNP.

### Single-tissue S-PrediXcan
S-PrediXcan was run using GTEx v8 elastic net models across six tissues, achieving 82–83% SNP utilization across all tissues. Bonferroni correction was applied within each tissue separately, with thresholds ranging from 5.88×10-6 (adipose subcutaneous, 8,503 genes tested) to 2.01×10-5 (uterus, 2,486 genes tested), reflecting the number of genes with trained prediction models in each tissue (Table 3).

Five of six tissues yielded at least one significant gene; vagina had no significant findings, which is expected given its small GTEx v8 sample size (N = 156 individuals).

The most gene-rich tissue was whole blood with 5 significant genes, followed by adipose subcutaneous (4 genes). Key single-tissue hits included CYP19A1 in adipose subcutaneous (Z = 6.33, P = 2.38×10-10), SKAP1 in whole blood (Z = −6.00, P = 1.93×10-9), and HEY2 exclusively in ovary (Z = −5.89, P = 3.97×10-9). Per-tissue Z-scores for the seven genes from Kho et al. (2021) revealed clear tissue specificity consistent with the published colocalization results: CYP19A1 was significant in adipose and blood but absent in reproductive tissues; HEY2 was testable only in ovary; and SKAP1 showed significance only in whole blood with a near-zero Z-score in uterus (Z = 0.50). GLDN did not reach significance in any tissue despite being the top S-MultiXcan hit, with its strongest signal in adipose subcutaneous (Z = 3.07, P = 0.002). This illustrates that the joint-tissue approach is needed for genes with modest but consistent cross-tissue signals.

### Joint-tissue S-MultiXcan
S-MultiXcan jointly tested 13,180 genes across all six tissues, applying a single Bonferroni threshold of P < 3.79×10-6 (0.05 / 13,180 genes tested), identifying nine significant genes (Table 4). All seven Kho et al. (2021) genes were replicated, with GLDN reaching joint-tissue significance (P = 1.02×10-12) despite no single-tissue Bonferroni hit. Two additional genes, SRP14 (P = 1.29×10-6) and TSEN2 (P = 1.29×10-6), also passed correction. Best-tissue assignments were consistent with single-tissue results: adipose subcutaneous drove GLDN, CYP19A1, SNX11, and TSEN2; whole blood drove SKAP1, EVI2A, and EEFSEC; and ovary was the sole tissue for HEY2.

Table 3. Elastic Net S-PrediXcan results per tissue.

| Tissue | Genes Tested | Bonferroni Threshold | Significant Genes |
|---|---:|---:|---:|
| Adipose Subcutaneous | 8,503 | 5.88×10-6 | 4 |
| Adipose Visceral Omentum | 7,206 | 6.94×10-6 | 3 |
| Whole Blood | 7,118 | 7.02×10-6 | 5 |
| Uterus | 2,486 | 2.01×10-5 | 1 |
| Ovary | 3,516 | 1.42×10-5 | 1 |
| Vagina | 2,510 | 1.99×10-5 | 0 |


Table 4. S-MultiXcan significant genes (Bonferroni threshold = 3.79×10-6; 13,180 genes tested).

| Gene | P-value | Tissues (n) | Indep. Components | Best Tissue | Z-range |
|---|---:|---:|---:|---|---|
| GLDN | 1.02×10-12 | 4 | 4 | Adipose Subcutaneous | −1.88 to 3.07 |
| CYP19A1 | 3.89×10-12 | 2 | 2 | Adipose Subcutaneous | 5.76 to 6.33 |
| HEY2 | 3.97×10-9 | 1 | 1 | Ovary | −5.89 |
| SKAP1 | 1.35×10-8 | 2 | 2 | Whole Blood | −6.00 to 0.50 |
| SNX11 | 5.92×10-7 | 4 | 2 | Adipose Subcutaneous | 3.37 to 5.34 |
| EVI2A | 1.19×10-6 | 3 | 2 | Whole Blood | −5.20 to −3.35 |
| SRP14 | 1.29×10-6 | 3 | 3 | Adipose Visceral Omentum | −2.02 to 2.70 |
| TSEN2 | 1.29×10-6 | 2 | 2 | Adipose Subcutaneous | −1.30 to 3.75 |
| EEFSEC | 1.94×10-6 | 3 | 3 | Whole Blood | 2.54 to 5.01 |


**Note:** SRP14 and TSEN2 are novel findings not reported in Kho et al. (2021).

### Mashr S-PrediXcan Results
S-PrediXcan was re-run using GTEx v8 mashr models across the same six tissues, achieving 69–70% SNP utilization, which is lower than elastic net (82–83%) due to mashr's use of varID format (chr_pos_ref_alt_b38) rather than rsIDs. Mashr models tested substantially more genes per tissue (10,090–12,293 vs. 2,486–8,503 for elastic net), reflecting broader gene coverage. Whole blood yielded the most significant genes with 5 genes, followed by vagina (6 genes) and ovary (5 genes) (Table 5). Notable mashr-specific findings included HNF1B in vagina (Z = 9.19, P = 4.04×10-20) and EIF2AK4 in vagina (Z = 5.05, P = 4.40×10-7), neither of which was significant under elastic net.

### Seven Genes from Kho et al. as Benmarking for GTEx Model Comparison
Comparing significant genes between models revealed that for several Kho et al. (2021) genes, elastic net and mashr identified different tissues as driving the association with endometrial cancer risk (Table 6). CYP19A1 showed the most notable shift: elastic net identified it in both adipose subcutaneous (Z = 6.33) and whole blood (Z = 5.76), while mashr showed a negative Z-score in adipose subcutaneous (Z = −1.49) and concentrated the signal in whole blood (Z = 6.61). 

HEY2 showed the most extreme disagreement between the two models due to mashr having no ovary model for HEY2, thus it could not test the tissue-gene pair. Elastic net had a trained prediction model for HEY2 in ovary and identified a strong signal (Z = −5.89, Bonferroni significant). This highlights a practical limitation of mashr models that higher overall gene coverage does not guarantee coverage of every biologically relevant gene-tissue combination. 

SKAP1 was Bonferroni significant in whole blood under elastic net (Z = −6.00) but only nominally associated under mashr (Z = −2.03). EEFSEC and EVI2A were directionally consistent across both models, with whole blood driving the signal in both cases.

At the tissue level, whole blood showed the highest overlap, with three genes significant in both models (CYP19A1, EEFSEC, EVI2A). Adipose subcutaneous, uterus, and ovary had zero shared significant genes between models (Table 7). SNX11 showed nearly identical Z-scores across all six tissues under mashr (Z = 4.42–4.43), with no tissue-to-tissue variation.

### Comparison to Kho et al. (2021)

Under elastic net, all seven Kho et al. (2021) S-MultiXcan genes were replicated with concordant best-tissue assignments (Table 6). Under mashr, tissue assignments diverged for several genes, most notably CYP19A1 (paper: adipose subcutaneous; mashr: whole blood) and HEY2 (paper: ovary; mashr: no ovary model available).

Table 5. Mashr S-PrediXcan results per tissue.

| Tissue | Genes Tested | Bonferroni Threshold | Significant Genes |
|---|---:|---:|---:|
| Adipose Subcutaneous | 12,293 | 4.07×10-6 | 4 |
| Adipose Visceral Omentum | 12,013 | 4.16×10-6 | 2 |
| Whole Blood | 10,432 | 4.79×10-6 | 5 |
| Uterus | 10,384 | 4.81×10-6 | 3 |
| Ovary | 10,966 | 4.56×10-6 | 5 |
| Vagina | 10,090 | 4.96×10-6 | 6 |

Table 6. Per-tissue Z-scores for Kho et al. (2021) genes under elastic net (EN) and mashr models.
Asterisk (*) indicates Bonferroni significance within that tissue and model. (—) indicates no trained prediction model available for that gene-tissue combination.

| Gene | Tissue | EN Z-score | Mashr Z-score |
|---|---|---:|---:|
| GLDN | Adipose SC | 3.07 | 0.85 |
| GLDN | Adipose Vis | −1.57 | — |
| GLDN | Whole Blood | — | −1.58 |
| GLDN | Uterus | −1.88 | −1.48 |
| GLDN | Ovary | −1.03 | −0.42 |
| GLDN | Vagina | — | −1.53 |
| CYP19A1 | Adipose SC | 6.33* | −1.49 |
| CYP19A1 | Adipose Vis | — | −1.49 |
| CYP19A1 | Whole Blood | 5.76* | 6.61* |
| CYP19A1 | Uterus | — | — |
| CYP19A1 | Ovary | — | — |
| CYP19A1 | Vagina | — | −1.49 |
| HEY2 | Adipose SC | — | −2.69 |
| HEY2 | Adipose Vis | — | 0.37 |
| HEY2 | Ovary | −5.89* | — |
| SKAP1 | Adipose SC | — | 1.89 |
| SKAP1 | Adipose Vis | — | −2.22 |
| SKAP1 | Whole Blood | −6.00* | −2.03 |
| SKAP1 | Uterus | 0.50 | −0.44 |
| SKAP1 | Ovary | — | 0.91 |
| SKAP1 | Vagina | — | 2.00 |
| SNX11 | Adipose SC | 5.34* | 4.42 |
| SNX11 | Adipose Vis | 5.13* | 4.42 |
| SNX11 | Whole Blood | 4.68* | 4.42 |
| SNX11 | Uterus | — | 4.42 |
| SNX11 | Ovary | 3.37 | 4.43 |
| SNX11 | Vagina | — | 4.43 |
| EVI2A | Adipose SC | −3.35 | −4.81* |
| EVI2A | Adipose Vis | −4.94* | −4.81* |
| EVI2A | Whole Blood | −5.20* | −4.81* |
| EVI2A | Ovary | — | −4.81* |
| EVI2A | Vagina | — | −4.81* |
| EEFSEC | Adipose SC | 2.54 | 3.59 |
| EEFSEC | Adipose Vis | 3.97 | 4.35 |
| EEFSEC | Whole Blood | 5.01* | 4.70* |
| EEFSEC | Uterus | — | 1.49 |
| EEFSEC | Ovary | — | 1.42 |
| EEFSEC | Vagina | — | 2.73 |

Table 7. Overlap of significant genes between elastic net and mashr models per tissue.

| Tissue | Elastic Net only | Both models | Mashr only |
|---|---|---|---|
| Adipose SC | AC145343.2, CYP19A1, RP5-890E16.5, SNX11 | — | ATF7IP2, CASC15, EVI2A, RP11-521C20.2 |
| Adipose Vis | RP5-890E16.5, SNX11 | EVI2A | EVI2B |
| Whole Blood | SKAP1, SNX11 | CYP19A1, EEFSEC, EVI2A | CBX1, EVI2B |
| Uterus | RP5-890E16.5 | — | RP11-521C20.2, RP11-521C20.5, SRP14-AS1 |
| Ovary | HEY2 | — | CYP3A7, EVI2A, EVI2B, RP11-521C20.2, RP11-521C20.5 |
| Vagina | — | — | EIF2AK4, EVI2A, EVI2B, HNF1B, RP11-521C20.2, SRP14-AS1 |

## Discussion
This project replicated and extended the multi-tissue TWAS analysis of endometrial cancer reported by Kho et al. (2021) using independently processed GWAS summary statistics and GTEx v8 prediction models. Across all three aims, the core findings of the original paper were reproduced, and several methodological insights emerged from the model comparison.

### Replication of Kho et al. (2021)
All seven genes reported in Kho et al. (2021) Table 1 were replicated under S-MultiXcan with concordant direction and best-tissue assignments. Six of seven genes also replicated at genome-wide significance (P < 5×10-8) at the GWAS locus level, with EEFSEC reaching only suggestive significance (P = 9.95×10-7) which is likely due to differences in GWAS processing or LD reference panels. The close agreement in p-values and tissue assignments between our pipeline and the original study supports the robustness of the S-MultiXcan framework for endometrial cancer TWAS.

Two novel genes, SRP14 and TSEN2, passed Bonferroni correction in our S-MultiXcan analysis but were not reported as primary findings in Kho et al. (2021). Both genes appear in the paper's Supplementary Table 7 at FDR < 0.01, suggesting they are borderline associations that our elastic net pipeline detected with slightly higher sensitivity. Whether these represent true endometrial cancer susceptibility genes warrants further investigation, including colocalization analysis and functional validation.

### Tissue Specificity and the Value of Multi-tissue TWAS
The per-tissue S-PrediXcan results highlighted the importance of tissue selection in TWAS. HEY2 was detectable only in ovary, consistent with the ovary-specific colocalization reported in Kho et al. (2021), while SKAP1 was exclusively significant in whole blood. CYP19A1, encodes aromatase and plays a central role in estrogen biosynthesis, showed strong signals in both adipose subcutaneous and whole blood. Those tissues are known to be major sites of peripheral estrogen production in postmenopausal women, which is the primary risk group for endometrial cancer.

GLDN is a particularly illustrative case for the value of joint-tissue methods as it did not reach Bonferroni significance in any single tissue (strongest signal: adipose subcutaneous, Z = 3.07, P = 0.002), yet it was the top S-MultiXcan hit (P = 1.02×10-12). This demonstrates that S-MultiXcan can recover genes whose eQTL signals are distributed across tissues and individually modest, providing statistical power that single-tissue methods cannot.

### Elastic Net vs. Mashr Model Comparison
The comparison between elastic net and mashr prediction models revealed meaningful differences in tissue assignments, despite broadly concordant effect directions for most genes. Mashr's cross-tissue shrinkage, which pools information across tissues via a shared prior, redistributed the CYP19A1 signal away from adipose subcutaneous and concentrated it in whole blood. Given that CYP19A1 expression in adipose tissue is biologically relevant to endometrial cancer through peripheral estrogen synthesis, the elastic net result, which preserves tissue-specific effects, may better reflect the underlying disease biology in this case. The most striking discrepancy was for HEY2, where elastic net identified a strong ovary-specific signal (Z = −5.89) that was entirely absent in mashr due to a missing ovary prediction model. This is not a statistical disagreement but a model coverage issue as mashr simply had no trained model for HEY2 in ovary, meaning that the tissue-specific signal was invisible to the mashr analysis. This highlights a practical limitation of mashr models that higher gene coverage overall does not guarantee coverage of biologically critical gene-tissue pairs. SNX11 showed an unusual pattern under mashr, with nearly identical Z-scores across all six tissues (Z = 4.42-4.43), compared to elastic net where the signal varied by tissue. This uniformity is likely a consequence of mashr's cross-tissue shrinkage pulling estimates towards a shared value, making it difficult to identify the tissue that is driving the association even when the gene-level signal is genuine.

Overall, these findings suggest that elastic net models better preserve tissue-specific eQTL architecture for endometrial cancer susceptibility genes, even though mashr is generally regarded as more statistically powerful for eQTL discovery. For TWAS applications where tissue prioritization is a primary goal, such as identifying the causal tissue for a disease association, the model choice has meaningful consequences beyond prediction accuracy.

### Limitations
Several limitations should be acknowledged. First, colocalization analysis (COLOC) was not performed, thus we cannot formally distinguish  pleiotropy from linkage at each identified locus. Without colocalization analysis, our results should be interpreted as gene prioritization rathen than causal inference.

Second, LD clumping was performed using 1000 Genomes EUR as the reference panel (N = 503), which provides less precise LD estimates than larger panels such as UK Biobank. UK Biobank individual-level genotype data was not used because it requires a formal data access agreement and is not publicly available. While the 16 independent loci identified are consistent with published EC GWAS findings, a larger reference panel would give more reliable clump boundaries.

Third, uterus and vagina have the smallest GTEx v8 sample sizes among the six tissues studied (N = 140 and N = 156, respectively). These are arguably the most biologically relevant tissues for endometrial cancer, yet they have the fewest genes with trained prediction models and the lowest statistical power. This limits the ability to detect uterus- or vagina-specific gene expression associations, and likely contributes to fewer finding of significant tissues despite their biological relevance.

Fourth, mashr S-MultiXcan was attempted but only recovered 4,047 genes which is substantially fewer than elastic net (13,180 genes) because the mashr SNP covariance file does not cover all genes present in the single-tissue mashr S-PrediXcan results. As a result, the model comparison in Aim 3 is based on single-tissue S-PrediXcan results rather than joint-tissue S-MultiXcan, which limits the direct comparability of the two models at the multi-tissue level.

## Conclusion 

This project replicated the multi-tissue TWAS findings of Kho et al. (2021) using an independently built pipeline, confirming all seven reported S-MultiXcan genes with concordant directions and tissue assignments. Two additional genes, SRP14 and TSEN2, were identified as borderline associations warranting further investigation. The comparison of elastic net and mashr prediction models demonstrated that model choice meaningfully affects tissue assignment. Elastic net model better preserved tissue-specific signals relevant to endometrial cancer biology, while mashr model cross-tissue shrinkage redistributed or obscured signals for several key genes. Together, these results shows the importance of both, tissue selection and prediction model choice in TWAS study design, and highlight the added power of joint-tissue methods for genes with modest but distributed eQTL signals.

## Methods

### GWAS Data
Harmonized summary statistics for endometrial cancer were downloaded from the NHGRI-EBI GWAS Catalog (study accession GCST006464; O'Mara et al. 2018), comprising of 12,906 cases and 108,979 controls of European ancestry (N = 121,885). The harmonized file was aligned to GRCh38.

### GWAS Quality Control
Quality control was implemented in Python 3.9.25 using pandas v2.2.3. Starting from 9,525,227 variants, the following filters were applied sequentially: removal of variants with missing values in any required field (chromosome, position, effect allele, other allele, beta, standard error, p-value, or rsID); restriction to autosomes (chromosomes 1–22); retention of biallelic SNPs only (both alleles in {A, T, C, G}); removal of palindromic SNPs (A/T and C/G pairs), which cannot be reliably strand-harmonized without allele frequency information; exclusion of the major histocompatibility complex region (chr6:26–34 Mb) due to complex LD structure, consistent with Kho et al. (2021); removal of duplicate chr:pos:allele combinations; and removal of variants with non-positive standard errors or p-values outside (0, 1]. After QC, 7,112,203 variants were retained for downstream analysis.

For each variant, a GTEx v8 PredictDB variant identifier was constructed in the format chr{N}{pos}{ref}_{alt}_b38, where the non-effect allele was treated as the reference allele. This format is required for matching against mashr prediction models. Elastic net models use rsIDs for SNP matching.

Genomic inflation was assessed using the lambda GC statistic, computed as the median chi-squared statistic derived from GWAS p-values via the chi-squared inverse survival function (df 1), divided by 0.4549, which is the expected median of a chi-squared distribution with one degree of freedom under the null.

### LD Clumping
Independent genome-wide significant loci were defined using PLINK v1.90b6.21 on the BU Shared Computing Cluster (SCC). The 1000 Genomes Phase 3 EUR samples (N = 503, GRCh38) were used as the LD reference panel, after removing duplicate variant IDs using PLINK2. Clumping parameters were: index SNP threshold P < 5×10-8, secondary SNP threshold P < 1×10-5, LD threshold r2 < 0.1, and a 500 kb window. Replication of published loci was assessed by identifying the most significant variant within the cytogenetic band of each of the seven genes reported in Kho et al. (2021) Table 1, using GRCh38 band boundaries obtained from the UCSC Genome Browser cytoband track.

### SNP Heritability
SNP heritability was estimated using LD Score Regression (LDSC v1.0.1) on the BU SCC. Summary statistics were first processed using munge_sumstats.py, filtering to HapMap3 SNPs. Heritability was estimated using Pan-UKBB European LD scores (UKBB.EUR.rsid). The analysis was run on the BU SCC using Python 2.7.16, as LDSC is incompatible with Python 3 and macOS ARM architecture.

### Transcriptome-Wide Association Study
Single-tissue TWAS was performed using S-PrediXcan (MetaXcan, Python 3.9.25) with GTEx v8 elastic net expression prediction models for six tissues: subcutaneous adipose, visceral omentum adipose, whole blood, uterus, ovary, and vagina. Models and covariance files were downloaded from PredictDB (https://predictdb.org/post/2021/07/21/gtex-v8-models-on-eqtl-and-sqtl/). SNP matching used rsIDs, achieving 82–83% model SNP utilization across tissues. Bonferroni correction was applied within each tissue separately (threshold = 0.05 / number of genes tested in that tissue). Joint multi-tissue TWAS was performed using S-MultiXcan with the elastic net SNP covariance matrix. A single Bonferroni threshold of P < 3.79×10-6 was applied (0.05 / 13,180 genes tested). The condition number cutoff was set to 30 as recommended by the MetaXcan documentation.

### Model Comparison
To assess the effect of prediction model choice on gene-tissue prioritization, S-PrediXcan was additionally run using GTEx v8 mashr prediction models (https://predictdb.org/post/2021/07/21/gtex-v8-models-on-eqtl-and-sqtl/). Mashr models use a multivariate adaptive shrinkage framework that borrows information across tissues, producing more regularized effect estimates. SNP matching for mashr models used the GTEx v8 varID format (chr_pos_ref_alt_b38), requiring the flags --model_db_snp_key varID and --keep_non_rsid, achieving 69–70% model SNP utilization. Significant genes from elastic net and mashr were compared per tissue to identify model-consistent versus model-specific associations. Per-tissue Z-scores for the seven Kho et al. (2021) genes were compared between models to assess tissue assignment concordance.

Mashr S-MultiXcan was also run, but the mashr cross‑tissue covariance file only produced results for 4,047 genes, compared with 13,180 genes for the elastic net S‑MultiXcan. This is because the mashr covariance file does not cover all genes that appear in the single‑tissue mashr S‑PrediXcan outputs. As a result, mashr S‑MultiXcan was not used in the main model comparison, and the elastic net versus mashr comparison is based only on the single‑tissue S‑PrediXcan results.


## References
Barbeira, A. N., Bonazzola, R., Gamazon, E. R., Liang, Y., Park, Y., Kim-Hellmuth, S., Wang, G., Jiang, Z., Zhou, D., Hormozdiari, F., Liu, B., Rao, A., Hamel, A. R., Pividori, M. D., Aguet, F., GTEx GWAS Working Group, Bastarache, L., Jordan, D. M., Verbanck, M., Do, R., … Im, H. K. (2021). Exploiting the GTEx resources to decipher the mechanisms at GWAS loci. Genome biology, 22(1), 49. https://doi.org/10.1186/s13059-020-02252-4

Gamazon, E. R., Wheeler, H. E., Shah, K. P., Mozaffari, S. V., Aquino-Michaels, K., Carroll, R. J., Eyler, A. E., Denny, J. C., GTEx Consortium, Nicolae, D. L., Cox, N. J., & Im, H. K. (2015). A gene-based association method for mapping traits using reference transcriptome data. Nature genetics, 47(9), 1091–1098. https://doi.org/10.1038/ng.3367

Barbeira, A.N., Dickinson, S.P., Bonazzola, R. et al. Exploring the phenotypic consequences of tissue specific gene expression variation inferred from GWAS summary statistics. Nat Commun 9, 1825 (2018). https://doi.org/10.1038/s41467-018-03621-1

Barbeira AN, Pividori M, Zheng J, Wheeler HE, Nicolae DL, et al. (2019) Integrating predicted transcriptome from multiple tissues improves association detection. PLOS Genetics 15(1): e1007889. https://doi.org/10.1371/journal.pgen.1007889

Bulik-Sullivan, B., Loh, PR., Finucane, H. et al. LD Score regression distinguishes confounding from polygenicity in genome-wide association studies. Nat Genet 47, 291–295 (2015). https://doi.org/10.1038/ng.3211

The GTEx Consortium ,The GTEx Consortium atlas of genetic regulatory effects across human tissues.Science369,1318-1330(2020).DOI:10.1126/science.aaz1776

Kent, W. J., Sugnet, C. W., Furey, T. S., Roskin, K. M., Pringle, T. H., Zahler, A. M., & Haussler, D. (2002). The human genome browser at UCSC. Genome Research, 12(6), 996–1006. https://doi.org/10.1101/gr.229102 

Kho, P.F., Wang, X., Cuéllar-Partida, G. et al. Multi-tissue transcriptome-wide association study identifies eight candidate genes and tissue-specific gene expression underlying endometrial cancer susceptibility. Commun Biol 4, 1211 (2021). https://doi.org/10.1038/s42003-021-02745-3

The 1000 Genomes Project Consortium. A global reference for human genetic variation. Nature 526, 68–74 (2015). https://doi.org/10.1038/nature15393

O’Mara, T.A., Glubb, D.M., Amant, F. et al. Identification of nine new susceptibility loci for endometrial cancer. Nat Commun 9, 3166 (2018). https://doi.org/10.1038/s41467-018-05427-7 

Pan-UKB team. (2020). Pan-UK Biobank. https://pan.ukbb.broadinstitute.org
Purcell, S., Neale, B., Todd-Brown, K., Thomas, L., Ferreira, M. A. R., Bender, D., Maller, J., Sklar, P., de Bakker, P. I. W., Daly, M. J., & Sham, P. C. (2007). PLINK: A tool set for whole-genome association and population-based linkage analyses. American Journal of Human Genetics, 81(3), 559–575. https://doi.org/10.1086/519795
